In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from pathlib import Path

from sklearn.linear_model import LinearRegression,Ridge, Lasso, ElasticNet
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler

In [ ]:
data_path = Path("/Users/patrykksiazek/Desktop/Machine-Learning---Preliminary-Data-Analysis/Data Preparation and Analysis/")

In [ ]:
grouped_train_logs = pd.read_csv(data_path / "grouped_train_logs.csv")
X_train_scores = pd.read_csv(data_path / "X_train_scores.csv")
X_test_scores = pd.read_csv(data_path / "X_test_scores.csv")

In [ ]:
grouped_train_logs

In [ ]:
cols_to_drop = ['Unnamed: 0', 'id', 'max_up_time', 'min_down_time', 'nonproduction_len_per_essay']

In [ ]:
train_data = grouped_train_logs.merge(X_train_scores, on="id").drop(columns=cols_to_drop)
test_data = grouped_train_logs.merge(X_test_scores, on="id").drop(columns=cols_to_drop)

In [ ]:
train_data

In [ ]:
X_train = train_data.drop(columns= ['score'])
y_train = train_data['score']
X_test = test_data.drop(columns= ['score'])
y_test = test_data['score']

# Benchmark - Constant Value

In [ ]:
dummy_regressor = DummyRegressor(strategy = 'mean')
dummy_regressor.fit(X_train, y_train)


In [ ]:
y_test_pred_dummy = dummy_regressor.predict(X_test)

In [ ]:
y_test_pred_dummy[0]

In [ ]:
rmse_test = np.sqrt(mean_squared_error(y_test, y_test_pred_dummy))


In [ ]:
rmse_test

In [ ]:
dummy_variance = np.var(y_test_pred_dummy - y_test)

In [ ]:
dummy_variance = np.var(y_test_pred_dummy - y_test)
dummy_bias = np.mean(y_test_pred_dummy) - np.mean(y_test)

In [ ]:
dummy_variance, dummy_bias

# Linear Regression

In [ ]:
def train_and_evaluate(model, X_train=X_train, X_test=X_test, y_train=y_train, y_test=y_test):

    model.fit(X_train, y_train)

    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    rmse_train = np.sqrt(mean_squared_error(y_train, y_train_pred))
    rmse_test = np.sqrt(mean_squared_error(y_test, y_test_pred))

    return rmse_train, rmse_test

In [ ]:
train_and_evaluate(dummy_regressor)[:2]

In [ ]:
model = LinearRegression()

In [ ]:
results = train_and_evaluate(model)

In [ ]:
results[:2]

In [ ]:
coef = model.coef_

In [ ]:
intercept = model.intercept_
intercept

In [ ]:
plt.bar(range(len(coef)), coef)
plt.xlabel("Indeks")
plt.ylabel("Wartość Współczynnika")

In [ ]:
scaler = StandardScaler()

X_train_normalized = scaler.fit_transform(X_train)
X_test_normalized = scaler.transform(X_test)

In [ ]:
results = train_and_evaluate(model, X_train_normalized, X_test_normalized)

In [ ]:
results [:2]

In [ ]:
coef = model.coef_

In [ ]:
intercept = model.intercept_
intercept

In [ ]:
plt.bar(range(len(coef)), coef)
plt.xlabel("Indeks")
plt.ylabel("Wartość Współczynnika")

In [ ]:
# Extracting the coefficient values
coef = model.coef_

# Sorting variables by the value of the coefficient
sorted_indices = coef.argsort()[::-1]
sorted_columns = X_train.columns[sorted_indices]
sorted_coef = coef[sorted_indices]

# Creating the plot
plt.figure(figsize=(10, 6))

# Visualizing the coefficients
plt.bar(sorted_columns, sorted_coef)
plt.xlabel('Zmienna')
plt.ylabel('Współczynnik coef')
plt.title('Wizualizacja coef współczynników regresji (posortowane)')

# Adding labels on the bars
for i, val in enumerate(sorted_coef):
    plt.text(i, val, f'{val: .2f}', ha='center', va='bottom')

# Displaying the plot
plt.xticks(rotation=90)
plt.show()

# Error Analysis

In [ ]:
y_test_pred = results[:2]
errors = y_test - y_test_pred

In [ ]:
np.var(errors)

In [ ]:
np.mean(y_test_pred) - np.mean(y_test)

In [ ]:
#Plotting the histogram of errors
plt.figure(figsize=(10,6))
plt.hist(errors, bins=30, edgecolor='k', alpha=0.7)
plt.title('Histogram of Prediction Errors')
plt.xlabel('Residuals(Actual - Predicted)')
plt.ylabel('Frequency')
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(10,6))
sns.scatterplot(x=y_test, y=y_test_pred)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--') # Diagonal line for reference
plt.title('Actual vs Predicted Score(Linear Regression)')
plt.xlabel('Actual Score')
plt.ylabel('Predicted Score')
plt.grid(True)
plt.show()

In [ ]:
error_df = pd.DataFrame ({"Score": y_test, 'Residuals': errors})

rmse_per_score = error_df.groupby('Score')['Residuals'].apply(lambda x: np.sqrt(np.mean(x['Residuals']**2)))
rmse_per_score.rename("RMSE", inplace=True)
rmse_per_score